# Real run on a Colab T4

Clone `main`, install, upload a labels-and-manifests bundle, fit the baselines, train AfroXLMR-base, evaluate, then download the SHA-256, the report, and the weights.

Use a GPU runtime. `configs/afroxlmr.yaml` freezes the embeddings and the bottom 8 of 12 layers, trains in fp16, and uses seed 42. It is sized for about 60-90 minutes.

Do not commit the upload or the outputs. This notebook stores no cell outputs.

In [ ]:
!git clone --branch main --single-branch \
  https://github.com/ChristopherKiokoStrathmore/-DSA8401---ML-CAPSTONE-PROJECT-.git /content/capstone
%cd /content/capstone
!python3 -m venv .venv
!make install

Upload one zip. It must contain a `labels/` directory and a `manifests/` directory.

`labels/` uses the canonical names in `docs/label_rubric.md`: `issue.csv`, `sentiment_hand.csv`, `sentiment_platform.csv`, `sentiment_test.csv`, `urgency_train.csv`, `urgency_test.csv`, `gemini_relabeled.csv`.

`manifests/` is the JSON from `make labels` (`issue.json`, `sentiment.json`, `urgency.json`, `registry.json`).

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

destination = Path('/content/data')
destination.mkdir(parents=True, exist_ok=True)
print('Upload the labels + manifests zip.')
uploaded = files.upload()
for name, payload in uploaded.items():
    path = destination / name
    path.write_bytes(payload)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as archive:
            archive.extractall(destination)
        print('extracted', path)
print('labels', list((destination / 'labels').glob('*.csv')))
print('manifests', list((destination / 'manifests').glob('*.json')))

In [ ]:
%cd /content/capstone
!make train-baselines LABELS=/content/data/labels MANIFESTS=/content/data/manifests KIND=data

In [ ]:
%cd /content/capstone
!make train-transformer LABELS=/content/data/labels MANIFESTS=/content/data/manifests KIND=data CONFIG=configs/afroxlmr.yaml

In [ ]:
%cd /content/capstone
!make evaluate LABELS=/content/data/labels MANIFESTS=/content/data/manifests KIND=data

In [ ]:
import hashlib
import shutil
from pathlib import Path
from google.colab import files

weights = Path('/content/capstone/artifacts/afroxlmr/model.pt')
digest = hashlib.sha256(weights.read_bytes()).hexdigest()
print(digest)
print(Path('/content/capstone/artifacts/afroxlmr/sha256.txt').read_text())
shutil.make_archive('/content/results', 'zip', '/content/capstone/reports')
shutil.make_archive('/content/weights', 'zip', '/content/capstone/artifacts')
files.download('/content/results.zip')
files.download('/content/weights.zip')